# Notebook 06: Fine-tuning & Alignment — Concepts Overview

**Time:** 20 minutes  
**Prerequisites:** Notebook 05 complete  
**Goal:** Understand what happens *after* pretraining — SFT, LoRA, DPO, and PPO

> **📌 Note:** Full hands-on implementation of fine-tuning is covered in a dedicated later class.
> This notebook builds **conceptual understanding** using Claude as your interactive guide.

## The Three-Stage Lifecycle

```
Stage 1: Pretraining         → Base model (predicts next token, no "personality")
    ↓
Stage 2: Supervised Fine-Tuning (SFT)  → Instruction-following model
    ↓  
Stage 3: Alignment (RLHF / DPO / PPO) → Helpful, harmless, honest model
```

You just built the data pipeline for **Stage 1** (notebooks 04–05). This notebook explains what happens in stages 2 and 3.

In [26]:
import os, sys, time
from pathlib import Path

notebook_dir = os.getcwd()
parent_dir   = str(Path(notebook_dir).parent)
if parent_dir not in sys.path:
    sys.path.insert(0, parent_dir)

from dotenv import load_dotenv
load_dotenv(os.path.join(parent_dir, '.env'))

from src.llm_client import LLMClient
from src.cost_tracker import CostTracker
from src.utils import format_response, append_to_reflection
import src.config as config

client  = LLMClient(path=config.PATH)
tracker = CostTracker()

outputs_dir = os.path.join('..', 'outputs')
os.makedirs(outputs_dir, exist_ok=True)

print("✅ Setup complete — ready for Notebook 06")

✓ Gemini client initialized (gemini-3-flash-preview)
✅ Setup complete — ready for Notebook 06


---

## Part 1: The Effect of SFT — Base vs Instruction-Tuned Models

A **base model** is trained purely to predict the next token. It will complete text in any way that's statistically consistent with its training data — which might not be what you want.

An **instruction-tuned** model (after SFT) has learned to respond helpfully to questions and commands.

In [27]:
print("=" * 65)
print("🧪 Experiment 1: Base Model vs Instruction-Tuned Model")
print("=" * 65)
print()

# Same prompt given to both a base-style and instruction-style prompt
neutral_prompt = "Explain the transformer architecture:"

# Base model style: just continue the text (complete mode)
base_style_system = None   # no system prompt = closer to base model behavior

# Instruction-tuned style: structured helpful response
instruction_style_system = (
    "You are a helpful ML engineering tutor. When asked to explain a concept, "
    "provide a clear, structured answer with: (1) one-sentence definition, "
    "(2) three key components, (3) one practical implication."
)

print(f"Prompt: \"{neutral_prompt}\"")
print()

print("--- Style A: No system prompt (base model behavior) ---")
resp_base = client.generate(
    prompt=neutral_prompt, system=None,
    max_tokens=400, temperature=1.0
)
if "error" not in resp_base:
    tracker.add_call(resp_base)
    print(resp_base['content'])

print()
print("--- Style B: With instruction-tuning system prompt ---")
resp_instr = client.generate(
    prompt=neutral_prompt, system=instruction_style_system,
    max_tokens=600, temperature=0.5
)
if "error" not in resp_instr:
    tracker.add_call(resp_instr)
    print(resp_instr['content'])

print()
print("💡 SFT training encodes the 'instruction-style' behavior into model weights,")
print("   so you don't need the system prompt to trigger it. The model just 'knows'.")

🧪 Experiment 1: Base Model vs Instruction-Tuned Model

Prompt: "Explain the transformer architecture:"

--- Style A: No system prompt (base model behavior) ---
The **Transformer** is a deep learning architecture introduced in 2017

--- Style B: With instruction-tuning system prompt ---

💡 SFT training encodes the 'instruction-style' behavior into model weights,
   so you don't need the system prompt to trigger it. The model just 'knows'.


---

## Part 2: What SFT Data Looks Like

SFT trains on **instruction-response pairs**. Here's the format used by major labs:

```json
{
  "instruction": "Summarize this paper abstract in one sentence.",
  "input": "We present a novel method for training large language models...",
  "output": "This paper introduces a new LLM training approach that..."
}
```

The quality of SFT data matters enormously — a small set of high-quality pairs beats a large set of mediocre ones.

In [7]:
import json
print("=" * 65)
print("🧪 Experiment 2: Generate Example SFT Training Pairs")
print("=" * 65)
print()

domain = "machine learning research"   # We'll ask Claude to generate pairs for this

sft_prompt = f"""
Generate 3 high-quality SFT (Supervised Fine-Tuning) training pairs for 
a language model assistant specialized in {domain}.

Each pair should follow this JSON format:
{{"instruction": "...", "input": "...", "output": "..."}}

Requirements for high quality:
- Instructions should be specific and actionable
- Outputs should be correct, helpful, and at appropriate length
- Cover different task types (explanation, summarization, comparison, generation)
- Inputs can be empty ("") if the instruction is self-contained

Return exactly 3 JSON objects, one per line.
"""

resp_sft = client.generate(
    prompt=sft_prompt,
    system="You are a data quality expert for LLM training. Generate realistic, high-quality training examples.",
    max_tokens=800,
    temperature=0.7
)

if "error" not in resp_sft:
    tracker.add_call(resp_sft)
    print("Generated SFT pairs:")
    print(resp_sft['content'])
else:
    print(f"❌ Error: {resp_sft['error']}")

🧪 Experiment 2: Generate Example SFT Training Pairs

Generated SFT pairs:
{"instruction": "Compare the Fréchet Inception Distance (FID) and Inception Score (IS) as evaluation metrics for generative models. Specifically, explain how each metric handles sample diversity and alignment with human perception.", "input": "", "output": "Fréchet Inception Distance (FID) and Inception Score (IS) are the two most prevalent metrics for evaluating GANs and diffusion models, but they operate on different principles:\n\n1. **Inception Score (IS):** It uses a pre-trained Inception-v3 network to classify generated images. It calculates the KL divergence between the conditional class distribution $p(y|x)$ and the marginal class distribution $p(y)$. A high IS indicates that the model produces images with clear objects (low entropy in $p(y|x)$) and high diversity across classes (high entropy in $p


### 🎯 TODO 1: Generate SFT Pairs for Your Domain

In [29]:
# TODO 1: Generate SFT training pairs for YOUR project domain

my_domain = "philanthropy"  # e.g., "medical diagnosis", "legal document review", "code debugging"

my_sft_prompt = f"""
Generate 5 high-quality SFT training pairs for an assistant specialized in {my_domain}.
Use the same format: {{"instruction": "...", "input": "...", "output": "..."}}
Make them realistic for what a practitioner in this field would actually ask.
"""

print("=" * 65)
print("🎯 TODO 1: My Domain SFT Pairs")
print("=" * 65)
print(f"Domain: {my_domain}")
print()

resp_todo1 = client.generate(
    prompt=my_sft_prompt,
    max_tokens=5000,
    temperature=0.7
)

if "error" not in resp_todo1:
    tracker.add_call(resp_todo1)
    print(resp_todo1['content'])


🎯 TODO 1: My Domain SFT Pairs
Domain: philanthropy

Here are 5 high-quality SFT training pairs tailored for a philanthropy-specialized assistant.

### 1. Strategy Development (Theory of Change)
**Instruction:** Help me develop a high-level Theory of Change (ToC) for a new grantmaking program focused on reducing the "summer melt" (the phenomenon where high school graduates who intend to go to college fail to enroll) in urban school districts.

**Input:** The program has a budget of $2M over three years. We want to fund a mix of non-profit interventions and school district pilot programs.

**Output:** To address the "summer melt," we will structure the Theory of Change around the core assumption that personalized outreach and administrative navigation support are the primary levers for enrollment.

**1. Inputs:** $2M funding, school district data sharing agreements, and partnerships with local education non-profits.
**2. Activities:** 
*   Funding "College Transition Coaches" to provide 

In [30]:

todo1_reflection = """


- What domain did you choose? Why is it interesting for fine-tuning?
  Philantrophy is an interesting domain for fine-tuning because it involves a wide range of tasks, 
  such as identifying effective charities, optimizing donation strategies, and understanding the impact of philanthropic efforts. 
  A language model assistant specialized in this area could provide valuable insights and recommendations to donors,
   non-profit organizations, and researchers in the field.`

- Looking at the generated pairs: what makes a GOOD training example vs. a bad one?
  A good training example has a clear and specific instruction, a relevant input (if needed), and a correct, helpful output that directly addresses the instruction. 
  It should also be realistic and representative of actual queries that practitioners in the domain might have. 
  A bad training example might have vague instructions, irrelevant inputs, or outputs that are incorrect, unhelpful, or too generic.


- How many high-quality pairs do you think you'd need to noticeably improve a base model?
  The number of high-quality pairs needed to noticeably improve a base model can vary widely depending on the complexity of the task, the quality of the examples, and the size of the base model.
  However, as a rough estimate, fine-tuning with a few hundred to a few thousand high-quality examples can often lead to noticeable improvements in specific tasks or domains. 
  For a significant improvement across a wide range of tasks within a domain, you might need tens of thousands of examples.   

  (Hint: GPT-3's InstructGPT used ~13,000 demonstrations)

  
"""
print()
print(todo1_reflection)





- What domain did you choose? Why is it interesting for fine-tuning?
  Philantrophy is an interesting domain for fine-tuning because it involves a wide range of tasks, 
  such as identifying effective charities, optimizing donation strategies, and understanding the impact of philanthropic efforts. 
  A language model assistant specialized in this area could provide valuable insights and recommendations to donors,
   non-profit organizations, and researchers in the field.`

- Looking at the generated pairs: what makes a GOOD training example vs. a bad one?
  A good training example has a clear and specific instruction, a relevant input (if needed), and a correct, helpful output that directly addresses the instruction. 
  It should also be realistic and representative of actual queries that practitioners in the domain might have. 
  A bad training example might have vague instructions, irrelevant inputs, or outputs that are incorrect, unhelpful, or too generic.


- How many high-qual

---

## Part 3: LoRA and Alignment — High-Level Mental Models

### LoRA (Low-Rank Adaptation)

Full fine-tuning updates all model weights — expensive for a 70B model. **LoRA** freezes the base model and trains tiny low-rank adapter matrices:

$$W' = W_0 + \Delta W = W_0 + BA$$

where $B \in \mathbb{R}^{d \times r}$ and $A \in \mathbb{R}^{r \times k}$, with rank $r \ll d$.

**Effect:** A 7B model with r=16 LoRA needs only ~4M trainable parameters (0.06%) instead of 7B!

### DPO vs PPO

Both are alignment methods — they ensure the model prefers helpful, harmless responses over harmful ones.

| | **DPO** (Direct Preference Optimization) | **PPO** (Proximal Policy Optimization) |
|---|---|---|
| **Approach** | Directly optimizes on preference pairs | Reinforcement learning with a reward model |
| **Complexity** | Simple — no reward model needed | Complex — requires training a separate reward model |
| **Stability** | More stable | Can be unstable (reward hacking) |
| **Used by** | Llama 2/3, Mistral, many open models | InstructGPT (GPT-3), GPT-4 (reportedly) |
| **Data needed** | (chosen, rejected) pairs | Human preference scores |

**DPO intuition:** Given a prompt, you have a "chosen" (good) response and a "rejected" (bad) response. DPO directly increases the model's probability of generating the chosen response and decreases the rejected one — no RL needed.

In [23]:
print("=" * 65)
print("🧪 Experiment 3: Ask Claude to Advise on Alignment")
print("=" * 65)
print()

# Use a specific project scenario to make this concrete
alignment_question = f"""
I'm building an AI assistant for {my_domain} that will be used by professionals.
I've collected 10,000 instruction-response pairs (SFT data).
After SFT, I want to align the model to be more helpful and less likely to generate
incorrect domain-specific information.

Please answer:
1. Should I use DPO or PPO? Why?
2. What should my preference pairs (chosen vs. rejected) look like for this domain?
3. How much alignment data do I realistically need?
4. What are the top 2 risks of misalignment in this specific domain?

Be specific and practical.
"""

resp_alignment = client.generate(
    prompt=alignment_question,
    system="You are a senior ML engineer with experience in fine-tuning and aligning language models for production use.",
    max_tokens=3600,
    temperature=0.3
)

if "error" not in resp_alignment:
    tracker.add_call(resp_alignment)
    print(format_response(resp_alignment, verbose=True))
else:
    print(f"❌ {resp_alignment['error']}")

🧪 Experiment 3: Ask Claude to Advise on Alignment

Model: gemini-3-flash-preview
Tokens: 163 in, 1032 out
Stop reason: FinishReason.STOP
As a senior ML engineer who has deployed alignment pipelines for high-stakes domains (finance, legal, and now philanthropy), here is my assessment for your AI assistant.

### 1. Should I use DPO or PPO?
**Recommendation: Use DPO (Direct Preference Optimization).**

**Why:**
*   **Stability and Simplicity:** PPO is notoriously difficult to tune. It requires maintaining four models in memory (policy, value, reference, and reward) and is highly sensitive to hyperparameters. DPO treats alignment as a classification problem, making it significantly more stable and computationally efficient.
*   **Data Efficiency:** For a domain-specific task with 10k SFT samples, you likely don't have the massive scale where PPO’s online learning advantages shine. DPO works exceptionally well on offline preference datasets.
*   **Factual Precision:** In philanthropy, "corr

### 🎯 TODO 2: Your Fine-Tuning Strategy Question

In [31]:
# TODO 2: Ask Claude a question about fine-tuning or alignment
#         that is directly relevant to YOUR project.

my_ft_question = """


Examples:
  - "What LoRA rank should I use for adapting qwen3.5:27b to [domain]?"
  - "How do I create DPO preference pairs when there's no clear 'wrong' answer?"
  - "Can I use LoRA to make the model respond in a different language?"
  - "What's the minimum GPU I need to fine-tune a 7B model with QLoRA?"
"""

print("=" * 65)
print("🎯 TODO 2: My Fine-Tuning Question")
print("=" * 65)
print()

resp_todo2 = client.generate(
    prompt=my_ft_question,
    system="You are an expert in LLM fine-tuning. Give a precise, practical answer.",
    max_tokens=3500,
    temperature=0.3
)

if "error" not in resp_todo2:
    tracker.add_call(resp_todo2)
    print(format_response(resp_todo2, verbose=False))


🎯 TODO 2: My Fine-Tuning Question

Here are precise, practical answers to your fine-tuning questions.

### 1. LoRA Rank ($r$) for Domain Adaptation
For adapting a model like **Qwen2.5-32B** (assuming 2.5, as 3.5 is not yet released) to a specific domain (e.g., Medical, Legal, or specialized Code):

*   **The Recommendation:** Use a rank **$r=64$** and **$\alpha=128$**.
*   **Why:** While $r=8$ or $16$ works for simple instruction-following, domain adaptation requires the model to learn new semantic relationships and terminology. Lower ranks often lack the "capacity" to store this new knowledge.
*   **Target Modules:** You must target **all linear layers** (e.g., `q_proj, k_proj, v_proj, o_proj, gate_proj, up_proj, down_proj`) to see significant domain shift. Targeting only attention layers is usually insufficient for domain-specific knowledge.

### 2. DPO Pairs with No Clear "Wrong" Answer
When both potential answers are factually correct, you must define "preference" based on **style,

In [32]:

todo2_reflection = """

- What did you ask and what did you learn?
  "What LoRA rank should I use for adapting qwen3.5:27b to Philanthropy?"
  - "How do I create DPO preference pairs when there's no clear 'wrong' answer?"
  - "Can I use LoRA to make the model respond in a different language?"
  - "What's the minimum GPU I need to fine-tune a 7B model with QLoRA?"
- How does this change your thinking about your project's technical approach?
    The answers to these questions will help me make informed decisions about the fine-tuning and alignment strategies for my project. 
    For example, understanding the appropriate LoRA rank can help me balance performance and resource constraints, while insights on creating DPO preference pairs can guide my approach to alignment when clear 'wrong' answers are not available. 
    Additionally, knowing whether I can use LoRA for language adaptation and the GPU requirements for fine-tuning will influence my technical planning and infrastructure choices.
- Which part of the fine-tuning lifecycle (SFT vs alignment) is most relevant to your project?
    Both SFT and alignment are relevant to my project, but alignment may be particularly crucial given the specialized nature of the domain (philanthropy) and the need to ensure that the model provides accurate and helpful information. 
    While SFT will help the model learn domain-specific knowledge, alignment will be essential to fine-tune the model's behavior and ensure it meets the specific needs and expectations of professionals in the philanthropy field.
 
"""
print()
print(todo2_reflection)




- What did you ask and what did you learn?
  "What LoRA rank should I use for adapting qwen3.5:27b to Philanthropy?"
  - "How do I create DPO preference pairs when there's no clear 'wrong' answer?"
  - "Can I use LoRA to make the model respond in a different language?"
  - "What's the minimum GPU I need to fine-tune a 7B model with QLoRA?"
- How does this change your thinking about your project's technical approach?
    The answers to these questions will help me make informed decisions about the fine-tuning and alignment strategies for my project. 
    For example, understanding the appropriate LoRA rank can help me balance performance and resource constraints, while insights on creating DPO preference pairs can guide my approach to alignment when clear 'wrong' answers are not available. 
    Additionally, knowing whether I can use LoRA for language adaptation and the GPU requirements for fine-tuning will influence my technical planning and infrastructure choices.
- Which part of t

## Summary & Reflection

In [33]:
full_reflection = f"""
### Training Lifecycle Understanding

The three stages:
1. Pretraining → base model (I built the data pipeline in notebooks 04-05)
2. SFT → instruction-tuned model (requires {my_domain} instruction-response pairs)
3. Alignment → DPO or PPO to prefer helpful/harmless outputs

### TODO 1 — SFT Data for My Domain ({my_domain})

{todo1_reflection.strip()}

### TODO 2 — Fine-tuning Strategy Question

Question asked: {my_ft_question.strip()[:200]}

{todo2_reflection.strip()}

### Key Takeaways

- LoRA reduces trainable parameters from billions to millions (0.06% of model size)
- DPO is simpler and more stable than PPO for most use cases
- High-quality SFT data (thousands) beats low-quality SFT data (millions)
- Alignment data: (chosen, rejected) pairs where chosen = preferred behavior
"""

reflection_file = append_to_reflection(
    notebook="06",
    section_title="Fine-tuning & Alignment Concepts",
    reflection_content=full_reflection,
    output_dir=os.path.join('..', 'outputs')
)
print(f"✅ Reflection saved: {reflection_file}")
print()
tracker.report()

✅ Reflection saved: ../outputs/homework_reflection.md

💰 API COST REPORT
Total API calls:     4
Total input tokens:  241
Total output tokens: 4,140
Total cost:          $0.0628

Last 4 calls:
  1. [07:54:11] 3 — 6in/16out — $0.0003
  2. [07:54:44] 3 — 59in/1587out — $0.0240
  3. [07:55:06] 3 — 57in/1580out — $0.0239
  4. [07:55:23] 3 — 119in/957out — $0.0147


## ✅ Notebook 06 Complete!

**What you accomplished:**
- ✅ Understood the base model vs instruction-tuned model difference
- ✅ Generated SFT training examples for your project domain
- ✅ Learned LoRA's parameter-efficiency mechanism
- ✅ Compared DPO vs PPO alignment approaches
- ✅ Got Claude's advice on your specific fine-tuning strategy

**Key resources for deeper study:**
- [LoRA paper](https://arxiv.org/abs/2106.09685) — Hu et al. 2021
- [DPO paper](https://arxiv.org/abs/2305.18290) — Rafailov et al. 2023
- [HuggingFace PEFT docs](https://huggingface.co/docs/peft/) — hands-on LoRA
- [TRL library](https://huggingface.co/docs/trl/) — SFT + DPO training (covered in later class)

**Next:** Open **Notebook 07: Test-Time Scaling** 🧠